In [1]:
!pip install sarvamai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.3/269.3 kB 9.9 MB/s eta 0:00:00


In [2]:
!pip install sarvamai openai-whisper jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.7 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a21d90c62f321cd3ccc46d995282d0f3d815693a67c1ba4dad5630dd8c7aaa6e
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [3]:
import os
import json
import whisper
import jiwer
from sarvamai import SarvamAI

# 1. Initialize the client (Remember to use your REGENERATED API key!)
client = SarvamAI(api_subscription_key="sk_69k5fwos_8b4Yj2OaevQGZkALwHOhh02f")

# 2. Define the requested directory structure
directories = [
    "sarvam-ai_hindi_outputs/male",
    "sarvam-ai_hindi_outputs/female",
    "sarvam-ai_bengali_outputs/male",
    "sarvam-ai_bengali_outputs/female"
]

for folder in directories:
    os.makedirs(folder, exist_ok=True)

# 3. Load Whisper medium model
print("Loading Whisper 'medium' model...")
whisper_model = whisper.load_model("medium")

# Custom transform to strip punctuation before calculating WER/CER
class IndicPunctuationRemover(jiwer.AbstractTransform):
    def process_string(self, s: str):
        return s.replace('।', '').replace(',', '').replace('?', '').replace('!', '').replace('"', '').replace("'", "")

eval_pipeline = jiwer.Compose([
    IndicPunctuationRemover(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords()
])

def evaluate_dataset(dataset_file, lang_code, base_output_dir):
    print(f"\n{'='*50}")
    print(f" Starting Evaluation for {lang_code} (Male & Female) ")
    print(f"{'='*50}")

    try:
        with open(dataset_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Error: '{dataset_file}' not found. Please upload it to Colab.")
        return

    # Handle dataset structures (List for Bengali, Dict for Hindi)
    if isinstance(data, list):
        sentences = [item.get("bengali_sentence") for item in data]
    else:
        sentences = [item.get("text") for item in data.values()]

    # We iterate through both genders for every single sentence
    genders = {
        "male": "shubh",
        "female": "ritu"
    }

    whisper_lang = "hi" if lang_code == "hi-IN" else "bn"

    for gender_name, speaker_id in genders.items():
        print(f"\n>>> Running Voice Profile: {gender_name.upper()} ({speaker_id}) <<<")

        total_wer, total_cer = 0.0, 0.0
        valid_sentences = 0
        target_dir = f"{base_output_dir}/{gender_name}"

        for i, reference_text in enumerate(sentences, start=1):
            if not reference_text:
                continue

            valid_sentences += 1
            file_name = f"{target_dir}/speech_{lang_code}_{i:02d}.wav"

            # A. Generate Audio via Sarvam AI
            try:
                response = client.text_to_speech.convert_stream(
                    text=reference_text,
                    target_language_code=lang_code,
                    speaker=speaker_id,
                    model="bulbul:v3",
                    pace=1.0,
                    speech_sample_rate=24000,
                )
                with open(file_name, "wb") as f:
                    for chunk in response:
                        f.write(chunk)
            except Exception as e:
                print(f" [{i:02d}] TTS Error: {str(e)}")
                continue

            # B. Transcribe via Whisper
            transcription_result = whisper_model.transcribe(file_name, language=whisper_lang)
            hypothesis_text = transcription_result["text"].strip()

            # C. Calculate Metrics
            # C. Calculate Metrics
            wer = jiwer.wer(reference_text, hypothesis_text, reference_transform=eval_pipeline, hypothesis_transform=eval_pipeline)
            cer = jiwer.cer(reference_text, hypothesis_text, reference_transform=eval_pipeline, hypothesis_transform=eval_pipeline)
            total_wer += wer
            total_cer += cer

        # Summary per Gender Profile
        if valid_sentences > 0:
            print(f"\n--- {lang_code} ({gender_name.upper()}) Metrics Summary ---")
            print(f"  Average WER: {total_wer / valid_sentences:.3f}")
            print(f"  Average CER: {total_cer / valid_sentences:.3f}")

# Execute the test routines
evaluate_dataset("hindi_evaluation_set.json", "hi-IN", "sarvam-ai_hindi_outputs")
evaluate_dataset("bengali_evaluation_set.json", "bn-IN", "sarvam-ai_bengali_outputs")

print("\n✓ Evaluation pipeline completed for both voice targets!")

Loading Whisper 'medium' model...


100%|██████████████████████████████████████| 1.42G/1.42G [00:14<00:00, 109MiB/s]



 Starting Evaluation for hi-IN (Male & Female) 
Error: 'hindi_evaluation_set.json' not found. Please upload it to Colab.

 Starting Evaluation for bn-IN (Male & Female) 

>>> Running Voice Profile: MALE (shubh) <<<

--- bn-IN (MALE) Metrics Summary ---
  Average WER: 1.089
  Average CER: 1.089

>>> Running Voice Profile: FEMALE (ritu) <<<

--- bn-IN (FEMALE) Metrics Summary ---
  Average WER: 1.029
  Average CER: 1.029

✓ Evaluation pipeline completed for both voice targets!


In [4]:
import os
from google.colab import files

# Define the folder you want to zip (change to "assamese_evaluation_outputs_f5" if needed)
folder_to_zip = "sarvam-ai_bengali_outputs"
zip_filename = f"{folder_to_zip}.zip"

# Zip the folder using system commands
print(f"Zipping {folder_to_zip}...")
os.system(f"zip -r {zip_filename} {folder_to_zip}")

# Trigger the download in your browser
print(f"Downloading {zip_filename}...")
files.download(zip_filename)

Zipping sarvam-ai_bengali_outputs...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>